# HW7: The Fairness Audit with a Lab-Measured Label

**Foundations of Data Science. Posted chapter 13, due with the final project. 100 points.**

Chapter 13 audited the screening model against *self-reported* diabetes and flagged door one: the label undercounts groups with less healthcare access. NHANES also measured glycohemoglobin (HbA1c) in the exam. The clinical definition of diabetes is **HbA1c ≥ 6.5%**. This assignment redoes the audit with the lab label and asks what changed.

Variables: `LBXGH` in `GHB_J` is HbA1c in percent. Join on `SEQN`; it's only measured for examined participants aged 12+.

**Rules.** Runs top to bottom from `data/raw/`. Adults 20+ with an HbA1c value. Same features, pipeline, split settings (`random_state=42`, stratified), and 80%-sensitivity cutoff rule as chapters 8 and 13. Written answers in the **Answer** cells.

## Setup

<!-- nb-links -->
[Download this notebook](hw7_fairness_hba1c.ipynb?download=1){download="hw7_fairness_hba1c.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/assignments/chapter_13/hw7_fairness_hba1c.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course_students"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/chapter_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("chapter_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq", "nhanes_ghb"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from core.datasets import load_raw

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
ghb = load_raw("nhanes_ghb")[["SEQN", "LBXGH"]].rename(columns={"LBXGH": "hba1c"})
df = nhanes.merge(ghb, on="SEQN", how="left")
adults = df[(df["age"] >= 20) & df["hba1c"].notna()].copy()
adults["sex"] = adults["sex"].map({1: "male", 2: "female"})
adults["race_eth"] = adults["race_eth"].map({1: "Mexican American", 2: "Other Hispanic", 3: "White", 4: "Black", 6: "Asian", 7: "Other/multi"})
adults["diabetes_lab"] = (adults["hba1c"] >= 6.5).astype(int)
print(adults.shape)
print("self-report prevalence:", adults["diabetes"].mean().round(3), "  lab prevalence:", adults["diabetes_lab"].mean().round(3))

## Q1. Two labels, one population (20 pts)

Cross-tabulate self-report vs lab. Then, by race/ethnicity and by sex: the share of lab-positive adults who self-report *no* diagnosis (the undiagnosed share). **Answer**: which groups have the highest undiagnosed share, and what that implies about a model trained on the self-report label.

In [ ]:
# Q1 code


**Answer (Q1):**



## Q2. Retrain on the lab label (20 pts)

Same pipeline and features as chapter 8 (age, bmi, waist_cm, sbp, dbp, income_ratio, sex, race_eth), logistic regression, C from `[0.01, 0.1, 1, 10]` on validation, target `diabetes_lab`. Report test AUC and the 80%-sensitivity cutoff chosen on validation. **Answer**: how AUC compares to the self-report model (0.80) and one reason it might differ.

In [ ]:
# Q2 code


**Answer (Q2):**



## Q3. The audit, lab label (30 pts)

`equiboots` by race/ethnicity and sex at your cutoff: sensitivity, FPR, PPV, predicted prevalence, AUC per group, plus disparity ratios vs White and vs female. Bootstrap (300 resamples) with Bonferroni-corrected tests on sensitivity, FPR, precision, AUC. **Answer**: which group gaps exist, which are significant, and how the picture compares to the chapter 13 audit on the self-report label.

In [ ]:
# Q3 code


**Answer (Q3):**



## Q4. Door one, measured (15 pts)

Score the **chapter 13 self-report model** on the test set but evaluate it against the **lab label**. Per-group sensitivity. **Answer**: for which groups does the self-report model miss the most lab-confirmed cases, and does that line up with the undiagnosed shares from Q1?

In [ ]:
# Q4 code


**Answer (Q4):**



## Q5. Recommendation (15 pts)

Under 200 words, to the clinic's medical director: which label the deployed model should be trained on, what changes in the screening rule, which group the self-report model was failing and by how much, and what the clinic should monitor after deployment.

**Answer (Q5):**

